# XGBoost Model Training

Tuning XGBoost's hyperparameters with Optuna (optimizing macro F1 on the
validation set), training the final model with the best parameters found,
and evaluating once on the untouched test set. Builds on
notebooks/02_model_selection.ipynb, where XGBoost was selected as the best
candidate with evidence.

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.preprocessing import LabelEncoder
import os

load_dotenv("../.env")

user = os.environ["POSTGRES_USER"]
password = os.environ["POSTGRES_PASSWORD"]
host = os.environ["POSTGRES_HOST"]
port = os.environ["POSTGRES_PORT"]
db = os.environ["POSTGRES_DB"]
engine = create_engine(f"postgresql+psycopg://{user}:{password}@{host}:{port}/{db}")

train_df = pd.read_sql("SELECT * FROM sensor_readings WHERE split = 'train'", engine)
val_df = pd.read_sql("SELECT * FROM sensor_readings WHERE split = 'val'", engine)

feature_cols = [c for c in train_df.columns if c not in
                ("id", "ahu_id", "reading_time", "labeling", "split")]

X_train = train_df[feature_cols]
X_val = val_df[feature_cols]

le = LabelEncoder()
y_train = le.fit_transform(train_df["labeling"])
y_val = le.transform(val_df["labeling"])

print(f"Train: {X_train.shape}, Val: {X_val.shape}")

Train: (105318, 18), Val: (35107, 18)


## Applying SMOTE to Valve position fault only

SMOTE cannot handle NaN, since it computes distances between real points
to generate synthetic ones, so structural nulls are temporarily filled
with a sentinel value before running it. Only Valve position fault is
oversampled (71 to 500 rows, via sampling_strategy), the one class thin
enough that a model can't reliably learn its pattern. The other four
classes already train well without oversampling, so they're left
untouched. Since every real Valve position fault row has NaN across all
heating columns (this fault only occurs while the AHU is in cooling
mode), synthetic rows are restored to true NaN there too, no invented
heating data. Applied to the training set only, validation and test stay
real, untouched data.

In [6]:
from imblearn.over_sampling import SMOTE

heating_cols = ["heating_supply_temp_1", "heating_supply_temp_2", "total_heating_pump",
                "heating_pump_1", "heating_pump_2", "heating_pump_3"]

X_train_for_smote = X_train.fillna(-999)

print("Before SMOTE:", pd.Series(y_train).value_counts().sort_index().values)

vpf_idx = le.transform(["Valve position fault"])[0]
smote = SMOTE(random_state=42, k_neighbors=5, sampling_strategy={vpf_idx: 500})
X_train_smote, y_train_smote = smote.fit_resample(X_train_for_smote, y_train)

X_train_smote = pd.DataFrame(X_train_smote, columns=feature_cols)

# Real rows (not synthetic) keep their exact original NaN positions -
# SMOTE with a targeted sampling_strategy never modifies non-VPF rows.
n_real = len(X_train)
X_train_smote.iloc[:n_real] = X_train.reset_index(drop=True)

# Synthetic VPF rows (appended after all real rows) get NaN in heating
# columns, since every real VPF row has NaN there too
X_train_smote.iloc[n_real:, X_train_smote.columns.get_indexer(heating_cols)] = np.nan

print("After SMOTE:", pd.Series(y_train_smote).value_counts().sort_index().values)
print(f"Real rows: {n_real}, synthetic rows appended: {len(X_train_smote) - n_real}")

Before SMOTE: [71640  4199   619 28789    71]
After SMOTE: [71640  4199   619 28789   500]
Real rows: 105318, synthetic rows appended: 429


In [7]:
# Confirm real rows kept their original NaN pattern exactly
print("NaN count in original X_train (heating_supply_temp_1):", X_train["heating_supply_temp_1"].isna().sum())
print("NaN count in first 105318 rows of X_train_smote:", X_train_smote.iloc[:105318]["heating_supply_temp_1"].isna().sum())

NaN count in original X_train (heating_supply_temp_1): 43922
NaN count in first 105318 rows of X_train_smote: 43922


## Hyperparameter search

Optuna runs a series of trials, each testing a different combination of
XGBoost hyperparameters, scored by macro F1 on the validation set. The
search space covers the settings that most affect tree-based model
performance: tree depth and count, learning rate, and regularization
strength.

In [8]:
import optuna
import xgboost as xgb
from sklearn.metrics import f1_score

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 1.0, log=True),
        "random_state": 42,
        "eval_metric": "mlogloss",
    }

    model = xgb.XGBClassifier(**params)
    model.fit(X_train_smote, y_train_smote)
    preds = model.predict(X_val)

    return f1_score(y_val, preds, average="macro")


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f"Best macro F1: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

[I 2026-09-27 11:38:51,539] A new study created in memory with name: no-name-982b50f3-0028-4772-af39-1ff493062e5d


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-27 11:38:56,904] Trial 0 finished with value: 0.9481564416695576 and parameters: {'n_estimators': 203, 'max_depth': 10, 'learning_rate': 0.28947308239293773, 'subsample': 0.6155762206142305, 'colsample_bytree': 0.8291348919099361, 'reg_alpha': 1.6800046202659104e-08, 'reg_lambda': 0.013758235435962413}. Best is trial 0 with value: 0.9481564416695576.
[I 2026-09-27 11:38:57,673] Trial 1 finished with value: 0.9395591283765162 and parameters: {'n_estimators': 62, 'max_depth': 3, 'learning_rate': 0.2677130652583378, 'subsample': 0.9309037499875892, 'colsample_bytree': 0.8177879269665236, 'reg_alpha': 1.2082472526728019e-08, 'reg_lambda': 0.22760228766064308}. Best is trial 0 with value: 0.9481564416695576.
[I 2026-09-27 11:39:02,003] Trial 2 finished with value: 0.9522868052341563 and parameters: {'n_estimators': 193, 'max_depth': 7, 'learning_rate': 0.017825583939741117, 'subsample': 0.7595943451277815, 'colsample_bytree': 0.7063568434357103, 'reg_alpha': 6.79866539444718e-07,

## Final model

Training XGBoost with Optuna's best parameters (macro F1 0.9915 on
validation, versus 0.9862 with default settings). This is the frozen model
used for the one-time test set evaluation.

In [9]:
best_params = study.best_params
best_params["random_state"] = 42
best_params["eval_metric"] = "mlogloss"

final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train_smote, y_train_smote)

final_model.save_model("../models/xgboost_fault_detector.json")

import json
class_names = list(le.classes_)
with open("../models/label_encoder_classes.json", "w") as f:
    json.dump(class_names, f)

print("Model saved. Classes:", class_names)

Model saved. Classes: ['Normal condition', 'Return air temperature fault', 'Supply air temperature fault', 'Supply fan fault', 'Valve position fault']


## Test set evaluation

This is the only time the test set is used. It was never seen during
training, validation, or hyperparameter tuning, so this result is the
honest, defensible measure of how the model generalizes.

In [ ]:
from sklearn.metrics import classification_report, f1_score

test_df = pd.read_sql("SELECT * FROM sensor_readings WHERE split = 'test'", engine)
X_test = test_df[feature_cols]
y_test = le.transform(test_df["labeling"])

test_preds = final_model.predict(X_test)

print(classification_report(y_test, test_preds, target_names=le.classes_))
print(f"Test macro F1: {f1_score(y_test, test_preds, average='macro'):.4f}")

                              precision    recall  f1-score   support

            Normal condition       1.00      1.00      1.00     23881
Return air temperature fault       0.99      0.99      0.99      1400
Supply air temperature fault       0.98      0.96      0.97       206
            Supply fan fault       1.00      1.00      1.00      9597
        Valve position fault       0.96      1.00      0.98        23

                    accuracy                           1.00     35107
                   macro avg       0.98      0.99      0.99     35107
                weighted avg       1.00      1.00      1.00     35107

Test macro F1: 0.9864


In [11]:
train_preds = final_model.predict(X_train)
train_f1 = f1_score(y_train, train_preds, average="macro")
val_f1 = f1_score(y_val, final_model.predict(X_val), average="macro")
test_f1 = f1_score(y_test, test_preds, average="macro")

print(f"Train macro F1: {train_f1:.4f}")
print(f"Val macro F1:   {val_f1:.4f}")
print(f"Test macro F1:  {test_f1:.4f}")

Train macro F1: 1.0000
Val macro F1:   0.9859
Test macro F1:  0.9864


## Test set evaluation

Evaluated once on the untouched test split (35,107 rows), never seen
during training, validation, or tuning.

| Class | Precision | Recall | F1 |
|---|---|---|---|
| Normal condition | 1.00 | 1.00 | 1.00 |
| Return air temperature fault | 0.99 | 0.99 | 0.99 |
| Supply air temperature fault | 0.98 | 0.96 | 0.97 |
| Supply fan fault | 1.00 | 1.00 | 1.00 |
| Valve position fault | 0.96 | 1.00 | 0.98 |

**Test macro F1: 0.9864.** All five fault classes score at or above 0.97
F1, including Valve position fault (500 SMOTE-augmented training examples,
23 real test examples), which achieves perfect recall and strong precision
on real, untouched test data.

**Checking for overfitting**:

| Split | Macro F1 |
|---|---|
| Train (real data only) | 1.0000 |
| Validation | 0.9859 |
| Test | 0.9864 |

Validation and test remain close to train and to each other. If the
model had overfit to the synthetic SMOTE examples, we would expect
validation or test to lag noticeably behind train, they don't, which
supports genuine generalization to real, unseen data rather than
memorization of synthetic training examples.